In [ ]:
import torch
import numpy as np
from eval_c2t import TwoSampleTest
from dm_ps.models.edm_wrapper import EDMOpenMRG, DecoderFreeEDMOpenMRG
from dm_ps.bridge import ddim

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


device = "cuda:1"
seed = 11235

torch.manual_seed(seed)
rng = np.random.RandomState(seed)




In [ ]:
# load model

# for the quantile-normalized data
network_pkl = "data/checkpoints/quantile-111997-relu.pkl"
model = DecoderFreeEDMOpenMRG(
    network_pkl=network_pkl, device=device, use_threshold=False
)

# for log-transformed data
# network_pkl = "log-transformed-060000-relu.pkl"
# model = EDMOpenMRG(network_pkl=network_pkl, device=device, use_threshold=False)

torch.cuda.empty_cache()



In [ ]:
n_samples = 5_000
x_shape = model.x_shape
initial_noise = torch.randn((n_samples, *x_shape), device=device)
samples = ddim(
    initial_noise,
    model.denoiser_fn,
    model.mk_sigmas_fn,
    n_steps=32,
    sigma_min=0.002,
    sigma_max=80.0,
    rho=7.0,
)
torch.cuda.empty_cache()

samples = samples.squeeze()



In [ ]:
n_selected_samples = min(5_000, n_samples)

gen_data = model.decode(samples).cpu()
gen_data = gen_data.squeeze()
gen_data = gen_data[:n_selected_samples]




In [ ]:
# load/select real data
real_data = np.load("data/train_rain_rate_f4.npy")
real_data = real_data[rng.choice(len(real_data), n_selected_samples, replace=False)]
real_data = torch.from_numpy(real_data).to(torch.float32)




In [ ]:
test = TwoSampleTest(device=device, seed=seed, n_epochs=200)

accuracy = test.compute_accuracy(real_data=real_data, generated_data=gen_data)

print(accuracy)


In [ ]:

# histogram train/test
# have a commune bins for comparisons
arr_bins = np.linspace(0, max(real_data.max(), gen_data.max()), num=50)

plt.hist(real_data.reshape(-1), label="real", bins=arr_bins, alpha=0.3)
plt.hist(gen_data.reshape(-1), label="gen", bins=arr_bins, alpha=0.3)
plt.yscale("log")

plt.ylim(0, 10**9)
plt.legend()
plt.xlabel("Rain intensity (mm/h)")




In [ ]:
# cumulative rain
cum_rain_gen = gen_data.sum(dim=(-2, -1))
cum_rain_real = real_data.sum(dim=(-2, -1))


arr_bins = np.linspace(0, max(cum_rain_real.max(), cum_rain_gen.max()), num=50)

plt.hist(cum_rain_real, label="real", bins=arr_bins, alpha=0.3)
plt.hist(cum_rain_gen, label="gen", bins=arr_bins, alpha=0.3)
plt.yscale("log")
plt.xlabel("Cumulative rain (mm/h)")

plt.ylim(0, 10**4)
plt.legend()



In [ ]:

from matplotlib.colors import LogNorm

arg_sorted_real = cum_rain_real.sort().indices
arg_sorted_gen = cum_rain_gen.sort().indices




In [ ]:
# uncurated samples

def _set_ax_layout(ax, rm_frame=False):
    ax.set_xticks([])
    ax.set_yticks([])
    rm_frame and ax.set_frame_on(False)


n_rows = 4
n_cols = 5
ratios_cols = [1 for _ in range(n_cols)]

fig, axes = plt.subplots(
    n_rows,
    ncols=2 * n_cols + 1,
    width_ratios=[*ratios_cols, 1 / 5, *ratios_cols],
    sharex=True,
    sharey=True,
    figsize=(8.27, 11.69 / 2.4),
)
fig.subplots_adjust(wspace=0.05, hspace=-0.3)

axes_gen = axes[:, :n_cols]
axes_gen[0, (n_cols // 2)].set_title("generated")

axes_real = axes[:, n_cols + 1 :]
axes_real[0, (n_cols // 2)].set_title("real")


# clean axes and rm col in the middle
[_set_ax_layout(ax) for ax in axes_gen.flatten()]
[_set_ax_layout(ax) for ax in axes_real.flatten()]
[_set_ax_layout(ax, rm_frame=True) for ax in axes[:, n_cols].flatten()]

idx_selected = rng.choice(n_selected_samples, n_cols * n_rows, replace=False)
for im, ax in zip(gen_data[idx_selected], axes_gen.flatten()):
    ax.imshow(im, norm=LogNorm(vmin=1e-2, vmax=100))

idx_selected = rng.choice(n_selected_samples, n_cols * n_rows, replace=False)
for im, ax in zip(real_data[idx_selected], axes_real.flatten()):
    ax.imshow(im, norm=LogNorm(vmin=1e-2, vmax=100))




In [ ]:
# 90% quantile samples

def _set_ax_layout(ax, rm_frame=False):
    ax.set_xticks([])
    ax.set_yticks([])
    rm_frame and ax.set_frame_on(False)


n_rows = 4
n_cols = 5
ratios_cols = [1 for _ in range(n_cols)]

fig, axes = plt.subplots(
    n_rows,
    ncols=2 * n_cols + 1,
    width_ratios=[*ratios_cols, 1 / 5, *ratios_cols],
    sharex=True,
    sharey=True,
    figsize=(8.27, 11.69 / 2.4),
)
fig.subplots_adjust(wspace=0.05, hspace=-0.3)

axes_gen = axes[:, :n_cols]
axes_gen[0, (n_cols // 2)].set_title("generated")

axes_real = axes[:, n_cols + 1 :]
axes_real[0, (n_cols // 2)].set_title("real")


# clean axes and rm col in the middle
[_set_ax_layout(ax) for ax in axes_gen.flatten()]
[_set_ax_layout(ax) for ax in axes_real.flatten()]
[_set_ax_layout(ax, rm_frame=True) for ax in axes[:, n_cols].flatten()]

threshold = min(200, n_selected_samples)
gen_quantile = gen_data[arg_sorted_gen[-threshold:]]
real_quantile = real_data[arg_sorted_real[-threshold:]]

idx_selected = rng.choice(threshold, n_cols * n_rows, replace=False)
for im, ax in zip(gen_quantile[idx_selected], axes_gen.flatten()):
    ax.imshow(im, norm=LogNorm(vmin=1e-2, vmax=100))


idx_selected = rng.choice(threshold, n_cols * n_rows, replace=False)
for im, ax in zip(real_quantile[idx_selected], axes_real.flatten()):
    ax.imshow(im, norm=LogNorm(vmin=1e-2, vmax=100))
